# P12 Dataset Scope Reproducible

由原 P12 拆出：產生 PO-D 資料範圍、原始文字欄位與生成覆蓋率相關表格。


## 0. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
from statistics import mean, median, stdev
from collections import Counter, defaultdict
from datetime import datetime
import csv
import json
import math
import re

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    scipy_stats = None
    SCIPY_AVAILABLE = False

BASE = Path("/home/ester/valid_description")
TGT = BASE / "experiment data" / "journal_figures_tables_20260629"
FIG = TGT / "figures"
TAB = TGT / "tables"
DER = TGT / "derived_audits"
SRC = TGT / "source_programs"
REPO = BASE / "text-conditioned-outfit-recommendation"
EXP_MAIN = REPO / "experiments"
EXP_NONE = REPO / "experiments_none_only"

for p in [FIG, TAB, DER, SRC]:
    p.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
WORD_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")

def read_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("[write]", path)

def display_csv(path: Path, max_rows: int = 12) -> None:
    rows = read_csv(path)
    for r in rows[:max_rows]:
        print(r)
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")

def fmt_float(x, nd=4) -> str:
    try:
        if x is None or math.isnan(float(x)):
            return ""
        return f"{float(x):.{nd}f}"
    except Exception:
        return ""

def fmt_pct(num, den) -> str:
    return f"{100 * float(num) / float(den):.2f}" if den else "0.00"

def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def to_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def query_token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text or "").strip()))

def word_tokens(text: str) -> list[str]:
    return [t.lower() for t in WORD_RE.findall(str(text or "").strip())]

def word_count(text: str) -> int:
    return len(word_tokens(text))

def original_field(original_data: dict, set_id: str, field: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get(field, "") or "").strip()
    return str(value or "").strip()

def original_url_name_plus_title(original_data: dict, set_id: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        parts = [str(value.get("url_name", "") or "").strip(), str(value.get("title", "") or "").strip()]
        return " ".join(p for p in parts if p).strip()
    return str(value or "").strip()

def generated_title_only(generated_data: dict, set_id: str) -> str:
    value = generated_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get("title", "") or "").strip()
    return str(value or "").strip()

print("BASE =", BASE)
print("TAB  =", TAB)
print("DER  =", DER)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)


BASE = /home/ester/valid_description
TAB  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables
DER  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits
SCIPY_AVAILABLE = True


## 1. Dataset Characterization and Evaluation Scope


In [2]:
# Correct scope:
# Main experiment uses experiments/.
# experiments_none_only and no_missing_none_ids.txt are supplementary ablation/control only.

split_dir = BASE / "polyvore_data/polyvore_outfits/disjoint"
train_records = read_json(split_dir / "train.json")
valid_records = read_json(split_dir / "valid.json")
test_records = read_json(split_dir / "test.json")
fitb_test_rows = read_json(split_dir / "fill_in_blank_test.json")
compat_test_rows = [line.strip().split() for line in (split_dir / "compatibility_test.txt").open(encoding="utf-8") if line.strip()]

original_title_path = BASE / "polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"
generated_title_path = BASE / "new_polyvore_outfit_titles.json"
original_titles = read_json(original_title_path)
generated_titles = read_json(generated_title_path)

train_n, valid_n, test_n = len(train_records), len(valid_records), len(test_records)
all_records = train_records + valid_records + test_records
all_split_ids = {str(r["set_id"]) for r in all_records}
generated_ids = {str(k) for k in generated_titles}

main_old_files = sorted(EXP_MAIN.glob("detail_cir_cir_old_featold_seed*.csv"))
main_new_files = sorted(EXP_MAIN.glob("detail_cir_cir_new_featnew_seed*.csv"))
if not main_old_files or not main_new_files:
    raise FileNotFoundError("Missing main experiments/ CIR detail files.")

cir_rows_per_seed = []
cir_pairs = set()
cir_fgs = set()
for fp in main_old_files:
    rows = read_csv(fp)
    cir_rows_per_seed.append(len(rows))
    for r in rows:
        cir_pairs.add((str(r["set_id"]), str(r["target_item_id"])))
        cir_fgs.add(str(r.get("target_item_fg", "")))

subset_ids_path = BASE / "no_missing_none_ids.txt"
subset_ids = {line.strip() for line in subset_ids_path.open(encoding="utf-8") if line.strip()} if subset_ids_path.exists() else set()
test_ids = {str(r["set_id"]) for r in test_records}
none_test_overlap = subset_ids & test_ids
none_cir_files = sorted((EXP_NONE / "detail_cir").glob("detail_cir_outfitUrlTitle_subset_none_subset_seed*.csv"))
none_cir_rows = [len(read_csv(fp)) for fp in none_cir_files]

outfit_lengths = [len(r.get("items", [])) for r in all_records]
item_ids = [str(item["item_id"]) for r in all_records for item in r.get("items", []) if item.get("item_id")]

def text_stats(texts: list[str]) -> dict:
    wc = [word_count(t) for t in texts]
    qc = [query_token_count(t) for t in texts]
    all_words = []
    for t in texts:
        all_words.extend(word_tokens(t))
    vocab = Counter(all_words)
    n = len(texts)
    empty = sum(1 for x in wc if x == 0)
    return {
        "n": n,
        "empty": empty,
        "pct_empty": fmt_pct(empty, n),
        "mean_words": mean(wc) if wc else 0,
        "median_words": median(wc) if wc else 0,
        "max_words": max(wc) if wc else 0,
        "mean_query_tokens": mean(qc) if qc else 0,
        "median_query_tokens": median(qc) if qc else 0,
        "max_query_tokens": max(qc) if qc else 0,
        "pct_le_3_words": fmt_pct(sum(1 for x in wc if x <= 3), n),
        "pct_le_5_words": fmt_pct(sum(1 for x in wc if x <= 5), n),
        "total_word_tokens": len(all_words),
        "unique_word_tokens": len(vocab),
        "type_token_ratio": (len(vocab) / len(all_words)) if all_words else 0,
    }

ordered_ids = sorted(generated_ids, key=lambda x: int(x) if x.isdigit() else x)
stat_original_title = text_stats([original_field(original_titles, sid, "title") for sid in ordered_ids])
stat_original_url_title = text_stats([original_url_name_plus_title(original_titles, sid) for sid in ordered_ids])
stat_generated = text_stats([generated_title_only(generated_titles, sid) for sid in ordered_ids])

cir_display = f"{cir_rows_per_seed[0]:,}" if len(set(cir_rows_per_seed)) == 1 else ", ".join(f"{x:,}" for x in cir_rows_per_seed)
none_display = f"{none_cir_rows[0]:,}" if none_cir_rows and len(set(none_cir_rows)) == 1 else ", ".join(f"{x:,}" for x in none_cir_rows)
main_total = train_n + valid_n + test_n
cir_skipped = len(fitb_test_rows) - len(cir_pairs)

t00 = [
    {"Category": "Dataset", "Statistic": "Source", "Value": "Polyvore Outfits Disjoint version (PO-D)"},
    {"Category": "Dataset split", "Statistic": "Train / validation / test / total outfits", "Value": f"{train_n:,} / {valid_n:,} / {test_n:,} / {main_total:,}"},
    {"Category": "Dataset split", "Statistic": "Generated-valid descriptions aligned to PO-D", "Value": f"{len(generated_ids):,} / {main_total:,} ({fmt_pct(len(generated_ids & all_split_ids), main_total)}%)"},
    {"Category": "Reference metadata", "Statistic": "Original Polyvore title metadata records", "Value": f"{len(original_titles):,} (metadata pool only; not main experimental N)"},
    {"Category": "CP evaluation", "Statistic": "Compatibility test pairs", "Value": f"{len(compat_test_rows):,}"},
    {"Category": "CP evaluation", "Statistic": "FITB test questions", "Value": f"{len(fitb_test_rows):,}"},
    {"Category": "CIR evaluation", "Statistic": "Initial test questions before CIR eligibility filtering", "Value": f"{len(fitb_test_rows):,}"},
    {"Category": "CIR evaluation", "Statistic": "Evaluable query-target pairs per seed", "Value": f"{cir_display} ({len(cir_pairs):,} unique pairs)"},
    {"Category": "CIR evaluation", "Statistic": "Excluded from CIR recall computation", "Value": f"{cir_skipped:,} test questions"},
    {"Category": "CIR evaluation", "Statistic": "Reason for 9,311 rather than 15,145", "Value": "CIR keeps only target fine-grained categories with an eligible same-category distractor pool in the original evaluation; skipped cases are not CP cases."},
    {"Category": "CIR evaluation", "Statistic": "Target fine-grained categories represented in CIR detail", "Value": f"{len(cir_fgs):,}"},
    {"Category": "CIR evaluation", "Statistic": "Seed-level paired CIR records across 5 seeds", "Value": f"{sum(cir_rows_per_seed):,}"},
    {"Category": "Supplementary no-missing ablation", "Statistic": "no_missing_none_ids total / test overlap", "Value": f"{len(subset_ids):,} / {len(none_test_overlap):,}"},
    {"Category": "Supplementary no-missing ablation", "Statistic": "CIR evaluable pairs per seed under experiments_none_only", "Value": f"{none_display or 'not found'} (not the main experiment scope)"},
    {"Category": "Outfit composition", "Statistic": "Item occurrences / unique items", "Value": f"{len(item_ids):,} / {len(set(item_ids)):,}"},
    {"Category": "Outfit composition", "Statistic": "Mean / median outfit length", "Value": f"{mean(outfit_lengths):.2f} / {median(outfit_lengths):.2f} items"},
]
write_csv(TAB / "T00_dataset_overview_po_d_main.csv", t00)

def t01_row(label, s, note):
    return {
        "Text field": label,
        "N records": f"{s['n']:,}",
        "Empty records": f"{s['empty']:,} ({s['pct_empty']}%)",
        "Mean / median / max word count": f"{s['mean_words']:.2f} / {s['median_words']:.2f} / {s['max_words']} words",
        "Mean query-token count": f"{s['mean_query_tokens']:.2f}",
        "Texts <= 3 words": f"{s['pct_le_3_words']}%",
        "Unique word tokens": f"{s['unique_word_tokens']:,}",
        "Type-token ratio": f"{s['type_token_ratio']:.4f}",
        "Note": note,
    }

t01 = [
    t01_row("Original title field", stat_original_title, "Dataset characterization only; original retrieval baseline uses url_name + title."),
    t01_row("Original url_name + title query (outfitUrlTitle)", stat_original_url_title, "Original-paper text embedding context used for fair baseline comparison."),
    t01_row("Generated context-aware description (title only)", stat_generated, "Proposed/generated query; original url_name is not appended."),
]
write_csv(TAB / "T01_text_field_comparison_po_d_main.csv", t01)

a01 = []
for label, s in [
    ("Original title field matched to PO-D ids", stat_original_title),
    ("Original url_name + title query matched to PO-D ids", stat_original_url_title),
    ("Generated context-aware title-only query", stat_generated),
]:
    a01.append({
        "dataset": label,
        "n_records": str(s["n"]),
        "empty_records": str(s["empty"]),
        "pct_empty": s["pct_empty"],
        "mean_word_count": f"{s['mean_words']:.4f}",
        "median_word_count": f"{s['median_words']:.4f}",
        "max_word_count": str(s["max_words"]),
        "mean_query_token_count": f"{s['mean_query_tokens']:.4f}",
        "median_query_token_count": f"{s['median_query_tokens']:.4f}",
        "max_query_token_count": str(s["max_query_tokens"]),
        "pct_word_len_le_3": s["pct_le_3_words"],
        "pct_word_len_le_5": s["pct_le_5_words"],
        "total_word_tokens": str(s["total_word_tokens"]),
        "unique_word_tokens": str(s["unique_word_tokens"]),
        "type_token_ratio": f"{s['type_token_ratio']:.6f}",
    })
write_csv(DER / "A01_title_length_audit.csv", a01)

a02 = [
    {"metric": "po_d_disjoint_train_records", "value": str(train_n), "source": str(split_dir / "train.json"), "notes": "Main PO-D split."},
    {"metric": "po_d_disjoint_valid_records", "value": str(valid_n), "source": str(split_dir / "valid.json"), "notes": "Main PO-D split."},
    {"metric": "po_d_disjoint_test_records", "value": str(test_n), "source": str(split_dir / "test.json"), "notes": "Main PO-D split."},
    {"metric": "po_d_disjoint_total_records", "value": str(main_total), "source": "disjoint train + valid + test", "notes": "Main dataset size."},
    {"metric": "generated_valid_title_records", "value": str(len(generated_titles)), "source": str(generated_title_path), "notes": "Aligned 1-to-1 with PO-D set_ids."},
    {"metric": "original_polyvore_title_metadata_records", "value": str(len(original_titles)), "source": str(original_title_path), "notes": "Reference metadata pool only."},
    {"metric": "cp_compatibility_test_pairs", "value": str(len(compat_test_rows)), "source": str(split_dir / "compatibility_test.txt"), "notes": "CP AUC evaluation scope."},
    {"metric": "cp_fitb_test_questions", "value": str(len(fitb_test_rows)), "source": str(split_dir / "fill_in_blank_test.json"), "notes": "CP FITB evaluation scope."},
    {"metric": "cir_evaluable_query_target_pairs_per_seed", "value": str(len(cir_pairs)), "source": str(EXP_MAIN / "detail_cir_cir_old_featold_seed*.csv"), "notes": "CIR recall scope after candidate-pool eligibility filtering."},
    {"metric": "cir_test_questions_excluded_by_cir_eligibility", "value": str(cir_skipped), "source": str(REPO / "CIR_evaluate.py"), "notes": "Skipped by CIR candidate-pool eligibility; not CP reassignment."},
    {"metric": "no_missing_none_ids_total", "value": str(len(subset_ids)), "source": str(subset_ids_path), "notes": "Supplementary no-missing ablation/control only."},
]
write_csv(DER / "A02_generation_coverage_audit.csv", a02)

display_csv(TAB / "T00_dataset_overview_po_d_main.csv", max_rows=20)


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T00_dataset_overview_po_d_main.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T01_text_field_comparison_po_d_main.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A01_title_length_audit.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A02_generation_coverage_audit.csv
{'Category': 'Dataset', 'Statistic': 'Source', 'Value': 'Polyvore Outfits Disjoint version (PO-D)'}
{'Category': 'Dataset split', 'Statistic': 'Train / validation / test / total outfits', 'Value': '16,995 / 3,000 / 15,145 / 35,140'}
{'Category': 'Dataset split', 'Statistic': 'Generated-valid descriptions aligned to PO-D', 'Value': '35,140 / 35,140 (100.00%)'}
{'Category': 'Reference metadata', 'Statistic': 'Original Polyvore title metadata records', 'Value': '68,306 (meta